# Pathogen → BioAssay Master Table

This notebook builds a unified table mapping:

**Pathogen → Taxonomy ID (TaxID) → BioAssay ID (AID)**  

This notebook:
1. Creates a Pathogen → Taxonomy ID (TaxID) table and dictionary


## 0. Setup

In [2]:
import pandas as pd
from pathlib import Path
import matplotlib.pyplot as plt
import numpy as np
import json
import requests

In [3]:
NOTEBOOK_DIR = Path().resolve()
DATA_RAW = NOTEBOOK_DIR.parent / "data" / "raw"
DATA_PROCESSED = NOTEBOOK_DIR.parent / "data" / "processed"
DATA_PROCESSED.mkdir(exist_ok=True)

In [4]:
pathogens = [
    "Acinetobacter baumannii", "Candida albicans", "Campylobacter",
    "Escherichia coli", "Enterococcus faecium", "Enterobacter",
    "Helicobacter pylori", "Klebsiella pneumoniae",
    "Mycobacterium tuberculosis", "Neisseria gonorrhoeae",
    "Pseudomonas aeruginosa", "Plasmodium falciparum",
    "Staphylococcus aureus", "Schistosoma mansoni",
    "Streptococcus pneumoniae"
]

## 01. Build Pathogen Taxonomy Table

### 01. Manually download pathogen summary
PubChem does not currently expose a stable API for retrieving organism-linked BioAssays directly from organism names. Therefore, the initial files used here were exported manually from:

**PubChem → Search → "Organism name" → Taxonomy →  Download: *Summary (Search Results)***

Each exported CSV (such as `PubChem_taxonomy_text_Acinetobacter baumannii.csv`) has been manually saved under `data\raw` and it contains:
- `Taxonomy_ID`
- `Taxonomy_Name`


In [8]:
abaumanii_summary = pd.read_csv(DATA_RAW / f"PubChem_taxonomy_text_Acinetobacter baumannii.csv")
abaumanii_summary.head(3)

,Taxonomy_ID,Data_Source,type,Taxonomy_Name,Synonyms,Linked_BioAssays,Linked_Proteins,Linked_Genes,Linked_Pathways,pmids,Identifiers,dois,pmcids,pclids,citations
0,575584,BioAssay|Patent|Cooccurrence,1,Acinetobacter baumannii ATCC 19606 = CIP 70.34...,Acinetobacter baumannii ATCC 19606 = CIP 70.34...,1368269|1368280|1368291|1657590,D0C9N6|D0C9N8,NaN,NaN,NaN,UniProt:575584|Wikidata:Q124691247,NaN,NaN,NaN,NaN
1,1116234,Cooccurrence,1,Acinetobacter baumannii AB5075,Acinetobacter baumannii AB5075|Acinetobacter b...,NaN,NaN,NaN,NaN,22374953,NaN,10.1128/jb.06749-11,PMC3294875,24892537,"Zurawski DV, Thompson MG, McQueary CN, Matalka..."
2,1116236,Cooccurrence,1,Acinetobacter baumannii AB5711,Acinetobacter baumannii AB5711|Acinetobacter b...,NaN,NaN,NaN,NaN,22374953,NaN,10.1128/jb.06749-11,PMC3294875,24892537,"Zurawski DV, Thompson MG, McQueary CN, Matalka..."


### 02. Keep only pathogen name and taxonomy
Create a single Pathogen with associated taxonomies table

In [9]:
def pathogen_taxid(pathogen: str) -> pd.DataFrame:
    filename = f"PubChem_taxonomy_text_{pathogen}.csv"
    filepath = DATA_RAW / filename

    df = pd.read_csv(filepath)
    df["Pathogen"] = pathogen

    return df[["Pathogen", "Taxonomy_ID", "Taxonomy_Name"]].drop_duplicates()

In [ ]:
pathogens_taxid_table = []

for pathogen in pathogens:
    print(f"Processing {pathogen}...")
    single_pathogen_taxid = pathogen_taxid(pathogen)
    pathogens_taxid_table.append(single_pathogen_taxid)

pathogens_taxid = pd.concat(pathogens_taxid_table, ignore_index=True)
pathogens_taxid = pathogens_taxid[["Pathogen", "Taxonomy_ID", "Taxonomy_Name"]]
pathogens_taxid

In [12]:
pathogens_taxid.to_csv(DATA_PROCESSED / "pathogens_taxid.csv", index=False)

### 03. Manually clean taxonomy table
Some taxonomy captures with PubChem query do not match the expected pathogens and have to be manually eliminated:

In [4]:
wrong_taxonomies = {
    "Acinetobacter baumannii": {
        "Acinetobacter calcoaceticus/baumannii complex",
    },
    "Candida albicans": {
        "Candida tropicalis",
    },
    "Campylobacter": {
        "Helicobacter pylori",
        "Helicobacter mustelae",
        "Aliarcobacter cryaerophilus",
        "Helicobacter cinaedi",
        "Helicobacter fennelliae",
        "Aliarcobacter butzleri",
        "Arcobacter nitrofigilis",
        "Helicobacter sp. CLO-3",
        "Firehammervirus CP220",
        "Firehammervirus CPt10",
        "Fletchervirus NCTC12673",
        "Fletchervirus CP81",
        "Fletchervirus CPX",
        "Firehammervirus CP21",
        "Fletchervirus CP30A",
        "Fletchervirus Los1",
    },
    "Escherichia coli": {
        "Tequintavirus AKFV33",
        "Enterobacteria phage CUS-3",
    },
    "Enterococcus faecium": {
        "Enterococcus casseliflavus",
    },
    "Enterobacter": {
        "Hafnia alvei",
        "Kosakonia radicincitans DSM 16656",
        "Kluyvera intermedia",
        "Cronobacter sakazakii",
        "Pluralibacter gergoviae",
        "Klebsiella aerogenes EA1509E",
        "Klebsiella aerogenes KCTC 2190",
        "Klebsiella aerogenes",
        "Pantoea agglomerans",
        "Lelliottia amnigena",
        "Lelliottia nimipressuralis",
        "Pluralibacter pyrinus",
        "Kosakonia radicincitans",
        "Siccibacter turicensis",
        "Franconibacter helveticus",
        "Franconibacter pulveris",
        "Kosakonia oryzae",
        "Kosakonia arachidis",
        "Kosakonia sacchari",
        "Kosakonia sacchari SP1",
        "Escherichia phage IME11",
        "Pluralibacter gergoviae ATCC 33028 = NBRC 105706",
        "Phytobacter massiliensis",
        "Atlantibacter hermannii",
        "Rahnella aquatilis",
        "Kosakonia cowanii",
        "Cronobacter sakazakii ATCC BAA-894",
        "Kosakonia oryzendophytica",
        "Kosakonia oryziphila",
        "Webervirus F20",
        "Franconibacter pulveris DSM 19144",
        "Hafnia phage Enc34",
        "Pseudenterobacter timonensis",
        "Karamvirus pg7",
        "Karamvirus cc31",
        "Slopekvirus eap3",
        "Eclunavirus EcL1",
        "Eapunavirus Eap1",
    },
    "Helicobacter pylori": {
        "Helicobacter mustelae",
    },
    "Klebsiella pneumoniae": {
        "Klebsiella michiganensis KCTC 1686",
        "Klebsiella variicola subsp. tropica",
    },
    "Mycobacterium tuberculosis": {
        "Mycobacterium avium",
        "Corynebacterium pseudotuberculosis",
    },
    "Pseudomonas aeruginosa": {
        "Pseudomonas virus Yua",
    },
    "Staphylococcus aureus": {
        "Dubowvirus dv11",
    },
}

In [7]:
# Remove rows with unwanted pathogens-taxonomy pairs
pathogens_taxid = pd.read_csv(DATA_PROCESSED / "pathogens_taxid.csv")

df = pathogens_taxid.copy()
keep = pd.Series(True, index=df.index)

for pathogen, taxonomy in wrong_taxonomies.items():
    # rows that match THIS pathogen AND have a name in THIS pathogen's bad set
    to_drop = (df["Pathogen"] == pathogen) & (df["Taxonomy_Name"].isin(taxonomy))
    
    # set those rows to False (drop them)
    keep &= ~to_drop

removed_pairs_df = df[keep].reset_index(drop=True)

In [8]:
# Remove rows with "phage" or "virus"
is_phage_or_virus = removed_pairs_df["Taxonomy_Name"].str.contains(
    r"phage|virus"
)

clean_df = removed_pairs_df[~is_phage_or_virus].reset_index(drop=True)

In [9]:
clean_df.to_csv(DATA_PROCESSED / "pathogens_taxid_cleaned.csv", index=False)

In [17]:
# Convert taxonomy table into dictionary
pathogens_taxid_cleaned_dict = (
    clean_df.groupby("Pathogen")["Taxonomy_ID"]
    .apply(list)
    .to_dict()
)

with open(DATA_PROCESSED / "pathogens_taxid_cleaned_dict.json", "w") as f:
    json.dump(pathogens_taxid_cleaned_dict, f, indent=2)

## 02. AIDs per pathogen using PubChem user interface (UI)

In [14]:
# Add UI (user interface) counts (manually annotated 17.11.25)
ui_counts = {
    "Acinetobacter baumannii": 15778,
    "Candida albicans": 23814,
    "Campylobacter": 622,
    "Escherichia coli": 63263,
    "Enterococcus faecium": 3864,
    "Enterobacter": 4023,
    "Helicobacter pylori": 1670,
    "Klebsiella pneumoniae": 11883,
    "Mycobacterium tuberculosis": 25323,
    "Neisseria gonorrhoeae": 1019,
    "Pseudomonas aeruginosa": 26093,
    "Plasmodium falciparum": 24519,
    "Staphylococcus aureus": 59672,
    "Schistosoma mansoni": 1276,
    "Streptococcus pneumoniae": 9474
}

AIDs_pathogen = (pd.DataFrame(ui_counts.items(), columns=["Pathogen", "UI_AIDs"])
    .sort_values("UI_AIDs", ascending=False)
    .reset_index(drop=True)
)

AIDs_pathogen

,Pathogen,UI_AIDs
0,Escherichia coli,63263
1,Staphylococcus aureus,59672
2,Pseudomonas aeruginosa,26093
3,Mycobacterium tuberculosis,25323
4,Plasmodium falciparum,24519
5,Candida albicans,23814
6,Acinetobacter baumannii,15778
7,Klebsiella pneumoniae,11883
8,Streptococcus pneumoniae,9474
9,Enterobacter,4023


## 03. AIDs per pathogen using PubChem Taxonomy search

PubChem does not currently expose a stable API for retrieving organism-linked BioAssays directly from organism names. Therefore, the initial files used here were exported manually from:

**PubChem → Search → "Organism name" → Taxonomy →  Download: *Summary (Search Results)***

Each exported CSV (such as `PubChem_taxonomy_text_Acinetobacter baumannii.csv`) has been manually saved under `data\raw`.

Each CSV contains a pipe-separated list of linked BioAssays. 

We expand these to one row per AID and annotate with the pathogen name.

Now, we will NOT filter out the wrong taxonomies we have manually filtered, in order to compare it with the UI_AIDs.

In [8]:
def taxid_aid(pathogen: str) -> pd.DataFrame:
    """Expand the PubChem Taxonomy→BioAssay mapping for a pathogen
    """

    df = pd.read_csv(DATA_RAW / f"PubChem_taxonomy_text_{pathogen}.csv")

    # Fill empty rows with empty strings "" (if no linked_bioassay, no rows for that pathogen to be counted)
    df["Linked_BioAssays"] = df["Linked_BioAssays"].fillna("")

    # Split into python list (from "1234|5678|9012" to ["1234", "5678", "9012"])
    df["AID"] = df["Linked_BioAssays"].astype(str).str.split("|")
    
    # Expand AIDs (one row per AID)
    df = df.explode("AID")

    df["Pathogen"] = pathogen

    return df[["Pathogen", "Taxonomy_ID", "Taxonomy_Name", "AID"]].drop_duplicates()

In [10]:
# Process all pathogens and create single table

pathogen_tables = []

for pathogen in pathogens:
    pathogen_table = taxid_aid(pathogen)
    pathogen_tables.append(pathogen_table)

all_pathogens = pd.concat(pathogen_tables, ignore_index=True)
all_pathogens = all_pathogens[["Pathogen", "Taxonomy_ID", "Taxonomy_Name", "AID"]]
all_pathogens.head()

,Pathogen,Taxonomy_ID,Taxonomy_Name,AID
0,Acinetobacter baumannii,575584,Acinetobacter baumannii ATCC 19606 = CIP 70.34...,1368269
1,Acinetobacter baumannii,575584,Acinetobacter baumannii ATCC 19606 = CIP 70.34...,1368280
2,Acinetobacter baumannii,575584,Acinetobacter baumannii ATCC 19606 = CIP 70.34...,1368291
3,Acinetobacter baumannii,575584,Acinetobacter baumannii ATCC 19606 = CIP 70.34...,1657590
4,Acinetobacter baumannii,1116234,Acinetobacter baumannii AB5075,


In [16]:
# Count number of AIDs per pathogen
taxid_counts = (
    all_pathogens.groupby("Pathogen")["AID"]
    .nunique()
    .reset_index(name="Taxonomy_AIDs")
    .sort_values("Taxonomy_AIDs", ascending=False)
    .reset_index(drop=True)
)

taxid_counts

,Pathogen,Taxonomy_AIDs
0,Mycobacterium tuberculosis,3809
1,Escherichia coli,3416
2,Candida albicans,3115
3,Pseudomonas aeruginosa,2415
4,Enterobacter,2258
5,Plasmodium falciparum,1968
6,Staphylococcus aureus,1897
7,Streptococcus pneumoniae,1585
8,Klebsiella pneumoniae,1536
9,Acinetobacter baumannii,1443


In [17]:
# Merge with previous AID counts
AIDs_pathogen = AIDs_pathogen.merge(
    taxid_counts,
    on="Pathogen",
    how="left"   # keep all pathogens from AIDs_pathogen
)

AIDs_pathogen

,Pathogen,UI_AIDs,Taxonomy_AIDs
0,Escherichia coli,63263,3416
1,Staphylococcus aureus,59672,1897
2,Pseudomonas aeruginosa,26093,2415
3,Mycobacterium tuberculosis,25323,3809
4,Plasmodium falciparum,24519,1968
5,Candida albicans,23814,3115
6,Acinetobacter baumannii,15778,1443
7,Klebsiella pneumoniae,11883,1536
8,Streptococcus pneumoniae,9474,1585
9,Enterobacter,4023,2258
